# 01. Load a model and generate

**Goal:** load `Qwen/Qwen2.5-1.5B-Instruct` in FP16 on RTX 4060 Ti (eGPU), generate one answer,
ad answer two questions:
1. How much VRAM does the model use?
2. Why is the first `generate()` call slower than the second?

**Setup (once, in a terminal, inside your venv)**
```bash
pip install torch --index-url https://download.pytorch.org/whl/cu124
pip install transformers accelerate jupyterlab pandas matplotlib
```

## 1. Pick the GPU *before* importing torch
`CUDA_DEVICE_ORDER=PCI_BUS_ID` makes CUDA number GPUs the same way as `nvidia-smi`
(0 = RTX 4070 Laptop, 1 = RTX 4060 Ti). `CUDA_VISIBLE_DEVICES="1"` then exposes **only** the
4060 Ti, so inside this notebook it becomes `cuda:0`. This avoids accidentally using the wrong card.

In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # 4060 Ti. Change to "0" to test the 4070 Laptop.

import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print(f"Total VRAM: {props.total_memory / 1024**3:.1f} GiB")

/home/thai-nguyen/thai/env/ai-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.14.1+cu130 | CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
Total VRAM: 15.6 GiB


## 2. Load tokenizer and model in FP16

In [2]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda:0"

torch.cuda.reset_peak_host_memory_stats()
t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map=DEVICE,
)
model.eval()
torch.cuda.synchronize()
load_s = time.perf_counter() - t0

n_params = sum(p.numel() for p in model.parameters())
weights_gib = torch.cuda.memory_allocated() / 1024**3
print(f"Load time: {load_s:.1f} s (first run includes download)")
print(f"Parameters: {n_params/1e9:.2f} B")
print(f"Expected FP16 weight size: {n_params*2/1024**3:.2f} GiB")
print(f"Allocated after load: {weights_gib:.2f} GiB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 216.53it/s]


Load time: 223.6 s (first run includes download)
Parameters: 1.54 B
Expected FP16 weight size: 2.88 GiB
Allocated after load: 2.88 GiB


## 3. Build a chat prompt
Instruct models are trained on a specific chat format. `apply_chat_template` turns a list of
messages into exactly that format. Print it once to see what the model really receives.

In [3]:
messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "Explain what a kubernetes pod is in three sentences"},
]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt_text)

inputs = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
print("Prompt tokens:", inputs["input_ids"].shape[1])

<|im_start|>system
You are a concise assistant.<|im_end|>
<|im_start|>user
Explain what a kubernetes pod is in three sentences<|im_end|>
<|im_start|>assistant

Prompt tokens: 30


## 4. Generate, twice, and time both calls

In [6]:
def generate_once(max_new_tokens=128):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens), elapsed

for i in range(2):
    text, n_tok, s = generate_once()
    print(f"Call {i + 1}: {n_tok} tokens in {s:.2f}s -> {n_tok/s:.1f} tok/s")
print("\nAnswer:\n", text)


Call 1: 51 tokens in 0.73s -> 69.8 tok/s
Call 2: 51 tokens in 0.72s -> 71.3 tok/s

Answer:
 A Kubernetes Pod is a group of one or more containers that work together as an isolated unit within the Kubernetes cluster, sharing resources and communicating with each other over a network. It provides a lightweight containerized application environment for deploying and managing applications at scale.


## 5. VRAM usage

In [7]:
peak = torch.cuda.max_memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3
print(f"Weights (allocated after load): {weights_gib:.2f} GiB")
print(f"Peak allocated during generate: {peak:.2f} GiB")
print(f"Reserved by PyTorch cache: {reserved:.2f} GiB")
print("Compare with `nvidia-smi`. It shows more, because it also counts the CUDA context (~300-500 MB).")

Weights (allocated after load): 2.88 GiB
Peak allocated during generate: 2.89 GiB
Reserved by PyTorch cache: 2.94 GiB
Compare with `nvidia-smi`. It shows more, because it also counts the CUDA context (~300-500 MB).


## 6. Write down answers

**Q1. VRAM:** weights ≈ params × 2 bytes in FP16. Peak is higher because of activations and the
**KV cache**, which grows with prompt + generated length.
- Weights (allocated after load): 2.88 GiB
- Peak allocated during generate: 2.89 GiB
- Reserved by PyTorch cache: 2.94 GiB
Compare with `nvidia-smi`: 3.18GiB. It shows more, because it also counts the CUDA context (~300-500 MB).

**Q2. Why is the first call slower?** Typical reasons: CUDA/cuBLAS kernels are loaded and tuned
lazily on first use, memory allocator warm-up, and Python-side lazy init. This is why every
benchmark needs **warm-up runs** that are not measured.
- Call 1: 51 tokens in 0.73s -> 69.8 tok/s
- Call 2: 51 tokens in 0.72s -> 71.3 tok/s
